# **TA-3.4 – Pipeline ETL Proyecto**
## **Predicción de Ventas Minoristas utilizando PySpark**

**Estudiante:**
Marlon Savier Torres Soto

# **Objetivo**

Diseñar e implementar un pipeline ETL distribuido utilizando Apache Spark (PySpark) sobre el dataset de ventas de Corporación Favorita. El pipeline permitirá extraer, validar, transformar y almacenar los datos en formato Parquet particionado, dejando el conjunto de datos preparado para el entrenamiento de modelos predictivos en la siguiente fase del proyecto.

# **Descripción del proyecto**

El proyecto tiene como finalidad desarrollar una solución de Big Data para analizar millones de registros históricos de ventas pertenecientes a Corporación Favorita. Mediante el uso de PySpark se busca procesar grandes volúmenes de información de forma distribuida, mejorar la calidad de los datos mediante un proceso ETL y generar un dataset optimizado para la construcción de modelos de Machine Learning orientados a la predicción de la demanda.

In [1]:
# INSTALACIÓN DE DEPENDENCIAS

!pip install pyspark -q

In [2]:
# IMPORTACIÓN DE LIBRERÍAS

import os
import time
import warnings
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import Window

import pyspark.sql.functions as F

from pyspark.sql.types import *

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import MinMaxScaler
from pyspark.ml.feature import StandardScaler

warnings.filterwarnings("ignore")

In [3]:
# CREACIÓN DE LA SESIÓN DE SPARK

spark = (
    SparkSession.builder
    .appName("Pipeline_ETL_Corporacion_Favorita")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("=======================================")
print("Spark inicializado correctamente")
print("Versión:", spark.version)
print("=======================================")

Spark inicializado correctamente
Versión: 4.0.3


In [4]:
# CONFIGURACIÓN GENERAL

inicio_pipeline = time.time()

print("Inicio del Pipeline ETL")

Inicio del Pipeline ETL


In [5]:
# FUNCIÓN PARA REGISTRAR CADA ETAPA DEL ETL

def log_etl(nombre, df_antes, df_despues):

    filas_antes = df_antes.count()
    filas_despues = df_despues.count()

    columnas_antes = len(df_antes.columns)
    columnas_despues = len(df_despues.columns)

    print("\n" + "="*70)
    print(f"ETAPA: {nombre}")
    print("="*70)

    print(f"Filas antes      : {filas_antes:,}")
    print(f"Filas después    : {filas_despues:,}")

    print(f"Columnas antes   : {columnas_antes}")
    print(f"Columnas después : {columnas_despues}")

    print(f"Filas modificadas: {filas_antes-filas_despues:,}")

    print("="*70)

In [6]:
# FUNCIÓN PARA MOSTRAR INFORMACIÓN GENERAL DEL DATASET

def resumen_dataset(df, nombre="Dataset"):

    print("\n")
    print("="*70)
    print(nombre)
    print("="*70)

    print(f"Filas    : {df.count():,}")
    print(f"Columnas : {len(df.columns)}")

    print("\nEsquema\n")
    df.printSchema()

    print("\nPrimeras filas\n")
    df.show(5, truncate=False)

# **Configuración del entorno**

Antes de iniciar el proceso ETL es necesario preparar el entorno de trabajo utilizando Apache Spark (PySpark), ya que el volumen del dataset supera los tres millones de registros y requiere procesamiento distribuido.

En esta etapa se inicializa una sesión de Spark, se importan las librerías necesarias y se definen funciones auxiliares que serán reutilizadas durante todo el pipeline.

Con el objetivo de facilitar la trazabilidad de las transformaciones, se implementa la función `log_etl()`, la cual permitirá registrar automáticamente el estado del dataset antes y después de cada transformación importante. De esta forma será posible evidenciar el impacto de cada etapa del proceso ETL, facilitando la validación de resultados y la documentación técnica del proyecto.

# **Inicio de la FASE E (Extract)**
## **Objetivo**

La fase de extracción tiene como finalidad obtener los datos desde las fuentes originales, validar su estructura y verificar su calidad antes de realizar cualquier transformación.

Para este proyecto se utilizará el dataset oficial de la competencia **Store Sales - Time Series Forecasting**, publicada por Corporación Favorita en Kaggle.

Los archivos principales corresponden a:

- train.csv
- stores.csv
- oil.csv

Posteriormente se integrarán mediante relaciones entre las columnas **store_nbr** y **date**, obteniendo un único DataFrame maestro que servirá como entrada para las siguientes fases del pipeline.

## **Validaciones de entrada**

Antes de transformar los datos se realizarán las siguientes verificaciones:

- Verificación de existencia de columnas obligatorias.
- Validación de tipos de datos.
- Conteo de registros.
- Identificación de valores nulos.
- Cálculo del porcentaje de nulos por columna.
- Identificación de registros duplicados.
- Revisión del esquema general del dataset.

Estas validaciones permiten detectar problemas de calidad desde el inicio del proceso ETL y evitar que errores de origen afecten el entrenamiento del modelo predictivo en fases posteriores.

In [8]:
# CARGA DE LOS ARCHIVOS CSV

TRAIN_PATH = "/content/train.csv"
STORES_PATH = "/content/stores.csv"
OIL_PATH = "/content/oil.csv"

print("Cargando archivos...")

train_df = spark.read.csv(
    TRAIN_PATH,
    header=True,
    inferSchema=True
)

stores_df = spark.read.csv(
    STORES_PATH,
    header=True,
    inferSchema=True
)

oil_df = spark.read.csv(
    OIL_PATH,
    header=True,
    inferSchema=True
)

print("Archivos cargados correctamente.")

Cargando archivos...
Archivos cargados correctamente.


In [9]:
# CONVERSIÓN DE FECHAS

train_df = train_df.withColumn(
    "date",
    F.to_date("date")
)

oil_df = oil_df.withColumn(
    "date",
    F.to_date("date")
)

In [10]:
# UNIFICACIÓN DEL DATASET

df = (
    train_df
    .join(stores_df, on="store_nbr", how="left")
    .join(oil_df, on="date", how="left")
)

df.cache()

print("Dataset unificado correctamente.")

Dataset unificado correctamente.


In [11]:
# RESUMEN GENERAL

resumen_dataset(df, "Dataset Maestro")



Dataset Maestro
Filas    : 1,110,449
Columnas : 11

Esquema

root
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- type: string (nullable = true)
 |-- cluster: integer (nullable = true)
 |-- dcoilwtico: double (nullable = true)


Primeras filas

+----------+---------+---+----------+-----+-----------+-----+---------+----+-------+----------+
|date      |store_nbr|id |family    |sales|onpromotion|city |state    |type|cluster|dcoilwtico|
+----------+---------+---+----------+-----+-----------+-----+---------+----+-------+----------+
|2013-01-01|1        |0  |AUTOMOTIVE|0.0  |0          |Quito|Pichincha|D   |13     |NULL      |
|2013-01-01|1        |1  |BABY CARE |0.0  |0          |Quito|Pichincha|D   |13     |NULL      |
|2013-01-01|1

## **FASE E — Validaciones del dataset**
### **Validación de la estructura del dataset**

Antes de aplicar cualquier transformación es indispensable comprobar que la información recibida cumple con la estructura esperada.

Una validación temprana permite detectar errores durante la etapa de extracción, evitando que estos se propaguen hacia las siguientes fases del pipeline.

Las verificaciones realizadas incluyen:

- existencia de columnas obligatorias;
- validación de tipos de datos;
- identificación de registros duplicados;
- análisis de valores nulos;
- cálculo del porcentaje de datos faltantes por columna.

Estas comprobaciones forman parte de las buenas prácticas en el diseño de procesos ETL para proyectos de Big Data.

In [12]:
# VALIDACIÓN DE COLUMNAS OBLIGATORIAS

columnas_esperadas = [

    "date",
    "store_nbr",
    "family",
    "sales",
    "onpromotion",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico"

]

columnas_actuales = df.columns

faltantes = [
    c
    for c in columnas_esperadas
    if c not in columnas_actuales
]

if len(faltantes) == 0:
    print("Todas las columnas esperadas están presentes.")
else:
    print("Faltan columnas:")
    print(faltantes)

Todas las columnas esperadas están presentes.


In [13]:
# TIPOS DE DATOS

print("Tipos de datos del DataFrame")

for nombre, tipo in df.dtypes:
    print(f"{nombre:20} {tipo}")

Tipos de datos del DataFrame
date                 date
store_nbr            int
id                   int
family               string
sales                double
onpromotion          int
city                 string
state                string
type                 string
cluster              int
dcoilwtico           double


In [14]:
# REPORTE DE NULOS

total = df.count()

reporte = []

for columna, tipo in df.dtypes:

    if tipo in ["double","float","int","bigint"]:

        nulos = df.filter(
            F.col(columna).isNull() |
            F.isnan(columna)
        ).count()

    else:

        nulos = df.filter(
            F.col(columna).isNull()
        ).count()

    porcentaje = (nulos / total) * 100

    reporte.append([

        columna,
        tipo,
        nulos,
        round(porcentaje,2)

    ])

reporte_nulos = pd.DataFrame(

    reporte,

    columns=[

        "Columna",
        "Tipo",
        "Nulos",
        "% Nulos"

    ]

)

reporte_nulos

,Columna,Tipo,Nulos,% Nulos
0,date,date,0,0.00
1,store_nbr,int,0,0.00
2,id,int,0,0.00
3,family,string,0,0.00
4,sales,double,0,0.00
5,onpromotion,int,0,0.00
6,city,string,0,0.00
7,state,string,0,0.00
8,type,string,0,0.00
9,cluster,int,0,0.00


In [17]:
# DUPLICADOS Y RESUMEN DE LA FASE EXTRACT

filas = df.count()

filas_unicas = df.distinct().count()

duplicados = filas - filas_unicas

print("="*60)

print(f"Filas totales     : {filas:,}")

print(f"Filas únicas      : {filas_unicas:,}")

print(f"Duplicados        : {duplicados:,}")

print("="*60)

display(

    reporte_nulos[
        reporte_nulos["Nulos"] > 0
    ]

)

Filas totales     : 1,110,449
Filas únicas      : 1,110,449
Duplicados        : 0


,Columna,Tipo,Nulos,% Nulos
10,dcoilwtico,double,343926,30.97


# **Inicio de la FASE T (Transform)**
## **T1 — Limpieza de duplicados y tratamiento de valores nulos**
Una vez validada la estructura del dataset, el siguiente paso consiste en mejorar la calidad de los datos eliminando registros duplicados y tratando los valores faltantes.

Durante el análisis exploratorio realizado en la fase anterior del proyecto se identificó que el dataset no presenta registros completamente duplicados; sin embargo, esta validación se ejecuta nuevamente como parte de las buenas prácticas en cualquier proceso ETL, garantizando la integridad de la información.

Respecto a los valores nulos, el principal problema se encuentra en la columna **dcoilwtico**, correspondiente al precio diario del petróleo. Estos valores faltantes no representan errores de captura, sino que ocurren porque el mercado petrolero no registra cotizaciones durante fines de semana y algunos días festivos.

Eliminar estos registros implicaría perder información valiosa sobre las ventas realizadas esos días. Por esta razón se aplicará una estrategia **Forward Fill**, propagando el último precio disponible hacia los días posteriores sin cotización.

Esta técnica conserva la continuidad temporal del dataset y evita introducir sesgos innecesarios en el modelo predictivo.

In [18]:
# DATAFRAME ORIGINAL

df_t1_antes = df

In [19]:
# ELIMINACIÓN DE DUPLICADOS

df = df.dropDuplicates()

print("Duplicados eliminados correctamente.")

Duplicados eliminados correctamente.


In [20]:
# FORWARD FILL PARA EL PRECIO DEL PETRÓLEO

window_oil = (
    Window
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, 0)
)

df = df.withColumn(
    "dcoilwtico",
    F.last(
        "dcoilwtico",
        ignorenulls=True
    ).over(window_oil)
)

In [21]:
# VALIDACIÓN DE NULOS DESPUÉS DEL FORWARD FILL

nulos_restantes = df.filter(
    F.col("dcoilwtico").isNull()
).count()

print(f"Nulos restantes en dcoilwtico: {nulos_restantes:,}")

Nulos restantes en dcoilwtico: 1,782


In [22]:
# LOG ETL

log_etl(

    "T1 - Limpieza de duplicados y tratamiento de nulos",

    df_t1_antes,

    df

)


ETAPA: T1 - Limpieza de duplicados y tratamiento de nulos
Filas antes      : 1,110,449
Filas después    : 1,110,449
Columnas antes   : 11
Columnas después : 11
Filas modificadas: 0


# **T2 — Corrección de tipos de datos y estandarización**
Los procesos ETL deben garantizar que todas las variables utilicen el tipo de dato adecuado antes de ser empleadas por algoritmos de análisis o aprendizaje automático.

Aunque durante la carga inicial PySpark infiere automáticamente los tipos de datos, se realiza una validación adicional para asegurar que las fechas permanezcan como DateType y que las columnas categóricas tengan un formato uniforme.

Además, se eliminan espacios innecesarios y se convierten los textos a mayúsculas para evitar inconsistencias ocasionadas por diferencias de escritura.

Finalmente, se corrigen posibles valores negativos en la variable **sales**. Aunque en el dataset original estos registros representan devoluciones de productos, para el objetivo de predicción de demanda se reemplazan por cero, evitando que introduzcan ruido en el entrenamiento del modelo.

In [23]:
# DATAFRAME ANTES DE LA TRANSFORMACIÓN

df_t2_antes = df

In [26]:
# ESTANDARIZACIÓN DE COLUMNAS DE TEXTO

columnas_texto = [

    "family",
    "city",
    "state",
    "type"

]

for columna in columnas_texto:

    df = df.withColumn(

        columna,

        F.upper(
            F.trim(
                F.col(columna)
            )
        )

    )
# VALIDACIÓN DEL TIPO DATE

df = df.withColumn(

    "date",

    F.to_date("date")

)

In [28]:
# CORRECCIÓN DE VENTAS NEGATIVAS

df = df.withColumn(
    "sales",
    F.when(
        F.col("sales") < 0,
        0
    ).otherwise(
        F.col("sales")
    )
)

# VERIFICACIÓN DEL ESQUEMA

df.printSchema()

# LOG ETL

log_etl(

    "T2 - Corrección de tipos y estandarización",

    df_t2_antes,

    df

)

root
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- type: string (nullable = true)
 |-- cluster: integer (nullable = true)
 |-- dcoilwtico: double (nullable = true)


ETAPA: T2 - Corrección de tipos y estandarización
Filas antes      : 1,110,449
Filas después    : 1,110,449
Columnas antes   : 11
Columnas después : 11
Filas modificadas: 0


# **T3 — Feature Engineering (Creación de nuevas variables)**
Una de las etapas más importantes de un proceso ETL consiste en generar nuevas variables que aporten mayor información al modelo de Machine Learning. Este proceso, conocido como **Feature Engineering**, permite representar mejor el comportamiento del negocio utilizando el conocimiento del dominio.

Para el proyecto de predicción de ventas minoristas se crearán nuevas variables derivadas de la fecha y de las promociones comerciales, ya que ambas influyen directamente en la demanda de productos.

Las variables creadas serán:

- **year:** año de la venta.
- **month:** mes de la venta.
- **day_of_week:** día de la semana.
- **is_weekend:** indica si la venta ocurrió durante un fin de semana.
- **sales_per_promotion:** relación entre las ventas y la cantidad de productos en promoción.

Estas variables proporcionan información temporal y comercial que no está explícitamente presente en el dataset original y pueden mejorar significativamente la capacidad predictiva del modelo.

In [29]:
# T3 - FEATURE ENGINEERING

df_t3_antes = df

# Variables temporales
df = (
    df
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)

# Variable binaria: fin de semana
df = df.withColumn(
    "is_weekend",
    F.when(
        F.col("day_of_week").isin([1, 7]),
        1
    ).otherwise(0)
)

# Variable derivada de promociones
df = df.withColumn(
    "sales_per_promotion",
    F.round(
        F.col("sales") /
        (F.col("onpromotion") + F.lit(1)),
        2
    )
)

print("Nuevas columnas creadas:")
print([
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sales_per_promotion"
])

df.select(
    "date",
    "sales",
    "onpromotion",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sales_per_promotion"
).show(10, truncate=False)

log_etl(
    "T3 - Feature Engineering",
    df_t3_antes,
    df
)

Nuevas columnas creadas:
['year', 'month', 'day_of_week', 'is_weekend', 'sales_per_promotion']
+----------+-----+-----------+----+-----+-----------+----------+-------------------+
|date      |sales|onpromotion|year|month|day_of_week|is_weekend|sales_per_promotion|
+----------+-----+-----------+----+-----+-----------+----------+-------------------+
|2013-01-02|57.0 |0          |2013|1    |4          |0         |57.0               |
|2013-01-02|0.0  |0          |2013|1    |4          |0         |0.0                |
|2013-01-02|38.0 |0          |2013|1    |4          |0         |38.0               |
|2013-01-02|997.0|0          |2013|1    |4          |0         |997.0              |
|2013-01-02|0.0  |0          |2013|1    |4          |0         |0.0                |
|2013-01-02|0.0  |0          |2013|1    |4          |0         |0.0                |
|2013-01-02|5.0  |0          |2013|1    |4          |0         |5.0                |
|2013-01-03|486.0|0          |2013|1    |5          |0 

# **T4 — Normalización de variables**
Los algoritmos de Machine Learning suelen obtener mejores resultados cuando las variables numéricas se encuentran en escalas similares. Si una variable presenta valores mucho mayores que otra, puede dominar el proceso de aprendizaje y afectar el rendimiento del modelo.

En este proyecto se utilizará **Min-Max Scaling**, ya que conserva la distribución original de los datos y transforma los valores al rango **[0,1]**, facilitando su utilización por distintos algoritmos predictivos.

La variable seleccionada para la normalización es **sales**, debido a que representa la variable objetivo principal del proyecto y presenta una alta dispersión entre sus valores mínimos y máximos.

In [30]:
# T4 - NORMALIZACIÓN (MIN-MAX)

df_t4_antes = df

# Vector de entrada requerido por Spark ML
assembler = VectorAssembler(
    inputCols=["sales"],
    outputCol="sales_vector"
)

df = assembler.transform(df)

# Escalamiento Min-Max
scaler = MinMaxScaler(
    inputCol="sales_vector",
    outputCol="sales_scaled"
)

scaler_model = scaler.fit(df)
df = scaler_model.transform(df)

print("Ejemplo de la variable normalizada:")

df.select(
    "sales",
    "sales_scaled"
).show(10, truncate=False)

log_etl(
    "T4 - Normalización Min-Max",
    df_t4_antes,
    df
)

Ejemplo de la variable normalizada:
+-----+-----------------------+
|sales|sales_scaled           |
+-----+-----------------------+
|57.0 |[0.001231873095459359] |
|0.0  |[0.0]                  |
|38.0 |[8.212487303062394E-4] |
|997.0|[0.02154697326619265]  |
|0.0  |[0.0]                  |
|0.0  |[0.0]                  |
|5.0  |[1.0805904346134729E-4]|
|486.0|[0.010503339024442957] |
|0.0  |[0.0]                  |
|0.0  |[0.0]                  |
+-----+-----------------------+
only showing top 10 rows

ETAPA: T4 - Normalización Min-Max
Filas antes      : 1,110,449
Filas después    : 1,110,449
Columnas antes   : 16
Columnas después : 18
Filas modificadas: 0


# **T5 — Clasificación del nivel de demanda (Transformación adicional)**
Como transformación adicional se construirá una nueva variable denominada **demand_level**, cuyo objetivo es clasificar el volumen de ventas de cada registro en distintos niveles de demanda.

Esta clasificación facilita el análisis del comportamiento comercial de las tiendas y permite identificar períodos de baja, media y alta demanda sin necesidad de revisar directamente los valores numéricos de ventas.

Para definir los rangos se utilizarán los percentiles del conjunto de datos, ya que representan una forma robusta de segmentar variables con distribuciones asimétricas como las ventas.

La nueva variable aporta información adicional que puede utilizarse en futuras tareas de análisis, segmentación de clientes, evaluación de campañas promocionales y entrenamiento de modelos de Machine Learning.

In [31]:
# T5 - CLASIFICACIÓN DEL NIVEL DE DEMANDA

df_t5_antes = df

# Cálculo de percentiles utilizando Spark

p25, p75 = df.approxQuantile(
    "sales",
    [0.25, 0.75],
    0.01
)

print(f"Percentil 25 : {p25:.2f}")
print(f"Percentil 75 : {p75:.2f}")

# Creación de la variable categórica

df = df.withColumn(
    "demand_level",
    F.when(
        F.col("sales") <= p25,
        "BAJA"
    )
    .when(
        F.col("sales") <= p75,
        "MEDIA"
    )
    .otherwise(
        "ALTA"
    )
)

print("\nDistribución del nivel de demanda:\n")

df.groupBy("demand_level") \
    .count() \
    .orderBy("demand_level") \
    .show()

log_etl(
    "T5 - Clasificación del nivel de demanda",
    df_t5_antes,
    df
)

Percentil 25 : 0.00
Percentil 75 : 121.00

Distribución del nivel de demanda:

+------------+------+
|demand_level| count|
+------------+------+
|        ALTA|278737|
|        BAJA|517460|
|       MEDIA|314252|
+------------+------+


ETAPA: T5 - Clasificación del nivel de demanda
Filas antes      : 1,110,449
Filas después    : 1,110,449
Columnas antes   : 18
Columnas después : 19
Filas modificadas: 0


# **Resumen de la Fase Transform**
Después de completar la fase de transformación, el dataset presenta una estructura más limpia, consistente y adecuada para el entrenamiento de modelos predictivos.

Las transformaciones implementadas fueron:

| Transformación | Objetivo |
|----------------|----------|
| **T1** Limpieza de duplicados y tratamiento de nulos | Mejorar la calidad del conjunto de datos eliminando duplicados e imputando el precio del petróleo mediante Forward Fill. |
| **T2** Corrección de tipos y estandarización | Garantizar consistencia en fechas, variables categóricas y valores numéricos. |
| **T3** Feature Engineering | Incorporar nuevas variables temporales y comerciales que aportan información relevante al modelo predictivo. |
| **T4** Normalización | Escalar la variable de ventas mediante Min-Max Scaling para facilitar el entrenamiento de algoritmos de Machine Learning. |
| **T5** Clasificación del nivel de demanda | Crear una nueva variable categórica que resume el comportamiento de las ventas en niveles de demanda. |

# **FASE L — LOAD**
Una vez completadas todas las transformaciones del proceso ETL, el dataset debe almacenarse en un formato optimizado para Big Data.

Se utilizará el formato **Apache Parquet**, ya que ofrece almacenamiento columnar, compresión eficiente y un acceso mucho más rápido que los archivos CSV tradicionales.

Además, se empleará compresión **Snappy**, ampliamente utilizada en Apache Spark debido a su excelente equilibrio entre velocidad de lectura, velocidad de escritura y reducción del tamaño de los archivos.

Finalmente, el dataset será particionado utilizando la columna **year**, lo que permitirá que futuras consultas y modelos de Machine Learning lean únicamente la información necesaria, reduciendo considerablemente los tiempos de procesamiento.

In [32]:
# FASE LOAD

import shutil

RUTA_PARQUET = "/content/dataset_etl_parquet"

# Eliminar carpeta previa si existe
if os.path.exists(RUTA_PARQUET):
    shutil.rmtree(RUTA_PARQUET)

inicio_escritura = time.time()

(
    df.write
      .mode("overwrite")
      .option("compression", "snappy")
      .partitionBy("year")
      .parquet(RUTA_PARQUET)
)

tiempo_escritura = time.time() - inicio_escritura

print("Dataset almacenado correctamente.")
print(f"Tiempo de escritura: {tiempo_escritura:.2f} segundos")

Dataset almacenado correctamente.
Tiempo de escritura: 38.64 segundos
